# Double Pendulum

Generalised coordinates :
$$
P(\theta_1,\theta_2)
$$
Cartesian coordinates (pendulum 1) :
$$
\begin{cases}
x_1=l_1\sin\theta_1\\
y_1=-l_1\cos\theta_1
\end{cases}
$$
Cartesian coordinates (pendulum 2) :
$$
\begin{cases}
x_2=l_1\sin\theta_1+l_2\sin\theta_2\\
y_2=-(l_1\cos\theta_1+l_2\cos\theta_2)
\end{cases}
$$
---

Newton's Laws :
$$
\begin{align}
(m_1+m_2)l_1\ddot{\theta}_1+m_2l_2\ddot{\theta}_2\cos(\theta_2-\theta_1)&=m_2l_2\dot{\theta}_2^2\sin(\theta_2-\theta_1)-(m_1+m_2)g\sin\theta_1\\
l_2\ddot{\theta}_2+l_1\ddot{\theta}_1\cos(\theta_2-\theta_1)&=-l_1\dot{\theta}_1^2\sin(\theta_2-\theta_1)-g\sin\theta_2
\end{align}
$$
In matrix form :
$$
A\,\vec{\ddot{\theta}}=\vec{b}
$$
$$
\text{where}\quad
\begin{cases}
\begin{align*}
\vec{\ddot{\theta}}&=\begin{bmatrix}
\ddot{\theta}_1\\
\ddot{\theta}_2
\end{bmatrix},\\\\
A &= \begin{bmatrix}
(m_1+m_2)l_1 & m_2l_2\cos(\theta_2-\theta_1) \\
l_1\cos(\theta_2-\theta_1) & l_2
\end{bmatrix},\\\\
\vec{b} &= \begin{bmatrix}
m_2l_2\dot{\theta}_2^2\sin(\theta_2-\theta_1)-(m_1+m_2)g\sin\theta_1\\
-l_1\dot{\theta}_1^2\sin(\theta_2-\theta_1)-g\sin\theta_2
\end{bmatrix}.
\end{align*}
\end{cases}
$$
---

## Eulers

In [43]:
import numpy as np

# Create complex matrix
def construct_matrix(y, m1, m2, l1, l2, g):
    """
    Returns the derivatives of the double pendulum system.
    """
    # Parameters
    θ1, θ2, ω1, ω2 = y # state vector
    delta = θ2 - θ1
    m12 = m1 + m2
    
    # Defining the vectors as matrix functions
    A = np.array([[m12 * l1, m2 * l2 * np.cos(delta)],
                  [l1 * np.cos(delta), l2]])
    
    b = np.array([
        m2 * l2 * abs(ω2**2) * np.sin(delta) - m12 * g * np.sin(θ1),
        -l1 * abs(ω1**2) * np.sin(delta) - g * np.sin(θ2)
    ])
    
    return A, b
    
# Convert to Cartesian coordinates
def position_bob1(θ1, l1):
    x1 = l1 * np.sin(θ1)
    y1 = -l1 * np.cos(θ1)
    return x1, y1

def velocity_bob1(θ1, ω1, l1):
    vx1 = l1 * ω1 * np.cos(θ1)
    vy1 = l1 * ω1 * np.sin(θ1)
    return vx1, vy1

def position_bob2(x1, y1, θ1, θ2, l1, l2):
    x2 = x1 + l2 * np.sin(θ2)
    y2 = y1 - l2 * np.cos(θ2)
    return x2, y2

def velocity_bob2(vx1, vy1, θ2, ω2, l2):
    vx2 = vx1 + l2 * ω2 * np.cos(θ2)
    vy2 = vy1 + l2 * ω2 * np.sin(θ2)
    return vx2, vy2

In [48]:
from scipy.linalg import solve

# Parameters
m1 = 1
m2 = 1
l1 = 1
l2 = 1
g = 9.81

# Initial conditions
θ1 = 80 * np.pi / 180  # 90 degrees
θ2 = 35 * np.pi / 180  # 90 degrees
ω1 = 0
ω2 = 0
A, b = construct_matrix([θ1, θ2, ω1, ω2], m1, m2, l1, l2, g)
α = solve(A, b)

# Loop
i = 1
dt = 0.05
sec = 10

# Array for time t, angles θ and angular velocities ω
t_arr = np.arange(0, sec+dt, dt)
n = len(t_arr)
θ1_arr = np.zeros(n); θ1_arr[0] = θ1
θ2_arr = np.zeros(n); θ2_arr[0] = θ2
ω1_arr = np.zeros(n); ω1_arr[0] = ω1
ω2_arr = np.zeros(n); ω2_arr[0] = ω2

while i < n:
    # Update angular velocity, ω
    ω1 += α[0] * dt
    ω2 += α[1] * dt

    # Update angle, θ
    θ1 += ω1 * dt
    θ2 += ω2 * dt

    # Append new values
    θ1_arr[i], θ2_arr[i] = θ1, θ2
    ω1_arr[i], ω2_arr[i] = ω1, ω2

    # Update angular acceleeration for next iteration, α
    A, b = construct_matrix([θ1, θ2, ω1, ω2], m1, m2, l1, l2, g)
    α = solve(A, b)

    # Update iteration count
    i += 1
    
# Find the positions and velocities in Cartesian coordinates
x1, y1 = position_bob1(θ1_arr, l1)
vx1, vy1 = velocity_bob1(θ1_arr, ω1_arr, l1)
x2, y2 = position_bob2(x1, y1, θ1_arr, θ2_arr, l1, l2)
vx2, vy2 = velocity_bob2(vx1, vy1, θ2_arr, ω2_arr, l2)

---

## Runge-Kutta

In [5]:
import numpy as np
from scipy.linalg import solve

# Convert Polar to Cartesian coordinates
def position_bob1(θ1, l1):
    x1 = l1 * np.sin(θ1)
    y1 = -l1 * np.cos(θ1)
    return x1, y1

def velocity_bob1(θ1, ω1, l1):
    vx1 = l1 * ω1 * np.cos(θ1)
    vy1 = l1 * ω1 * np.sin(θ1)
    return vx1, vy1

def position_bob2(x1, y1, θ1, θ2, l1, l2):
    x2 = x1 + l2 * np.sin(θ2)
    y2 = y1 - l2 * np.cos(θ2)
    return x2, y2

def velocity_bob2(vx1, vy1, θ2, ω2, l2):
    vx2 = vx1 + l2 * ω2 * np.cos(θ2)
    vy2 = vy1 + l2 * ω2 * np.sin(θ2)
    return vx2, vy2

# Functions which simulate the double pendulum
def double_pendulum_derivatives(t, y, m1, m2, l1, l2, g):
    """
    Returns the derivatives of the double pendulum system.
    """
    # Parameters
    θ1, θ2, ω1, ω2 = y # unpack state vector
    delta = θ2 - θ1
    m12 = m1 + m2
    
    # Defining the vectors as matrix functions
    A = np.array([
        [m12 * l1, m2 * l2 * np.cos(delta)],
        [l1 * np.cos(delta), l2]
    ])
    
    b = np.array([
        m2 * l2 * abs(ω2**2) * np.sin(delta) - m12 * g * np.sin(θ1),
        -l1 * abs(ω1**2) * np.sin(delta) - g * np.sin(θ2)
    ])
    
    # Solve for acceleration
    α1, α2 = solve(A, b)
    
    return np.array([ω1, ω2, α1, α2]) # return derivatives

def rk4_step(f, t, y, dt, *args):
    """
    4th Order Runge-Kutta step
    """
    # Runge-Kutta returns state vectors from derivatives
    k1 = f(t, y, *args)
    k2 = f(t + dt/2, y + dt/2 * k1, *args)
    k3 = f(t + dt/2, y + dt/2 * k2, *args)
    k4 = f(t + dt, y + dt * k3, *args)

    return y + dt/6 * (k1 + 2*k2 + 2*k3 + k4)

def simulate_double_pendulum(θ1, θ2, ω1=0, ω2=0,  
                             m1=1, m2=1, 
                             l1=1, l2=1, 
                             t_max=10, dt=0.01,
                             g=9.81):
    """
    Simulate the double pendulum
    """
    # Initial state
    y = np.array([θ1, θ2, ω1, ω2])
    
    # Time array
    t_values = np.arange(0, t_max+dt, dt)
    
    # Store results
    results = np.zeros((len(t_values), 4))
    results[0] = y
    
    # Simulation loop
    for i in range(1, len(t_values)):
        results[i] = rk4_step(double_pendulum_derivatives, t_values[i-1], 
                             results[i-1], dt, m1, m2, l1, l2, g)
    
    return t_values, results

In [ ]:
t, r = simulate_double_pendulum(0, [np.pi/4, np.pi/4, 0, 0], 1, 1, 1, 1, 9.81)

ValueError: too many values to unpack (expected 2)

# Tkinter

In [4]:
import numpy as np
import tkinter as tk
import ttkbootstrap as ttk
from ttkbootstrap.constants import *
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.figure import Figure
from matplotlib.animation import FuncAnimation

class DoublePendulumApp:
    def __init__(self, root, duration=10, fps=30):
        self.root = root
        self.duration = duration
        self.fps = fps
        self.total_frames = duration * fps
        
        self.setup_gui()
        
        
    def setup_gui(self):
        self.root.title(f"Amazing Double Pendulum Simulation ({self.duration}s)")
        self.root.geometry("500x500")
        
        
        # 1. Controls frame
        control_frame = ttk.Labelframe(self.root, bootstyle="primary", text="Controls")
        control_frame.place(x=50, y=50, width=300, height=100)
        
        # A. Bob 1 Frame
        bob1_frame = ttk.Labelframe(control_frame, bootstyle="danger", text="Pendulum 1")
        bob1_frame.pack(side=tk.LEFT, padx=5, pady=5)
        
        button1 = ttk.Button(bob1_frame, bootstyle="info", text="Restart", command='helo')
        button1.pack(side=tk.TOP, anchor=tk.W, padx=10, pady=10)
        
        # tk.Button(bob1_frame, text="Pause/Resume", command='helo').pack(side=tk.TOP, anchor=tk.W, padx=10, pady=10)
        
        
if __name__ == "__main__":
    root = tk.Tk()
    app = DoublePendulumApp(root, duration=30, fps=60)  # 30 seconds at 60 FPS
    root.mainloop()

In [4]:
import numpy as np
import tkinter as tk
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib.figure import Figure
from matplotlib.animation import FuncAnimation

class DoublePendulumApp:
    def __init__(self, root, duration=30, fps=30):
        self.root = root
        self.duration = duration
        self.fps = fps
        self.total_frames = duration * fps
        
        # Precompute all pendulum positions
        self.t, self.theta1, self.theta2 = self.simulate_double_pendulum()
        
        self.setup_gui()
        self.setup_animation()
    
    def simulate_double_pendulum(self):
        """Precompute pendulum motion for fixed duration"""
        t = np.linspace(0, self.duration, self.total_frames)
        
        # Your double pendulum equations here
        # This is a simplified example - replace with your actual simulation
        theta1 = np.sin(2 * np.pi * 0.5 * t) * 0.8
        theta2 = np.sin(2 * np.pi * 0.7 * t + 1) * 1.2
        
        return t, theta1, theta2
    
    def setup_gui(self):
        self.root.title(f"Double Pendulum Simulation ({self.duration}s)")
        
        # Control frame
        control_frame = tk.Frame(self.root)
        control_frame.pack(pady=10)
        
        tk.Button(control_frame, text="Restart", command=self.restart).pack(side=tk.LEFT, padx=5)
        tk.Button(control_frame, text="Pause/Resume", command=self.toggle_pause).pack(side=tk.LEFT, padx=5)
        
        # Matplotlib figure
        self.fig = Figure(figsize=(6, 6))
        self.ax = self.fig.add_subplot(111)
        self.ax.set_xlim(-2.5, 2.5)
        self.ax.set_ylim(-2.5, 2.5)
        self.ax.set_aspect('equal')
        self.ax.grid()
        
        # Initialize plot elements
        self.line, = self.ax.plot([], [], 'o-', lw=2, markersize=8)
        self.trace, = self.ax.plot([], [], ',-', alpha=0.6, lw=1)
        self.time_text = self.ax.text(0.02, 0.95, '', transform=self.ax.transAxes)
        self.progress_text = self.ax.text(0.02, 0.90, '', transform=self.ax.transAxes)
        
        self.canvas = FigureCanvasTkAgg(self.fig, master=self.root)
        self.canvas.get_tk_widget().pack()
    
    def setup_animation(self):
        self.current_frame = 0
        self.paused = False
        
        # Convert to positions
        L1, L2 = 1.0, 1.0
        self.x1 = L1 * np.sin(self.theta1)
        self.y1 = -L1 * np.cos(self.theta1)
        self.x2 = self.x1 + L2 * np.sin(self.theta2)
        self.y2 = self.y1 - L2 * np.cos(self.theta2)
        
        self.animation = FuncAnimation(
            self.fig, self.update, frames=self.total_frames,
            interval=1000/self.fps, blit=True, repeat=False
        )
    
    def update(self, frame):
        if self.paused:
            return self.line, self.trace, self.time_text, self.progress_text
            
        self.current_frame = frame
        
        # Update pendulum
        self.line.set_data([0, self.x1[frame], self.x2[frame]], 
                          [0, self.y1[frame], self.y2[frame]])
        
        # Update trace (last 100 points)
        start_idx = max(0, frame - 100)
        self.trace.set_data(self.x2[start_idx:frame+1], 
                           self.y2[start_idx:frame+1])
        
        # Update text
        self.time_text.set_text(f'Time: {self.t[frame]:.2f}s')
        progress = (frame / self.total_frames) * 100
        self.progress_text.set_text(f'Progress: {progress:.1f}%')
        
        return self.line, self.trace, self.time_text, self.progress_text
    
    def toggle_pause(self):
        self.paused = not self.paused
    
    def restart(self):
        self.current_frame = 0
        self.animation.event_source.stop()
        self.animation = FuncAnimation(
            self.fig, self.update, frames=self.total_frames,
            interval=1000/self.fps, blit=True, repeat=False
        )

if __name__ == "__main__":
    root = tk.Tk()
    app = DoublePendulumApp(root, duration=30, fps=60)  # 30 seconds at 60 FPS
    root.mainloop()

bgerror failed to handle background error.
    Original error: can't invoke "event" command: application has been destroyed
    Error in bgerror: can't invoke "tk" command: application has been destroyed


In [1]:
import tkinter as tk
from tkinter import ttk
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import matplotlib.pyplot as plt
import numpy as np

root = tk.Tk()
root.title("Interactive 2x3 Grid with Sliders")
root.geometry("1000x700")

# Create main frame
main_frame = ttk.Frame(root, padding=10)
main_frame.pack(fill="both", expand=True)

# Grid configuration
for i in range(2):  # rows
    main_frame.rowconfigure(i, weight=1)
for j in range(3):  # columns
    main_frame.columnconfigure(j, weight=1)

# --- Create shared data ---
x = np.linspace(-5, 5, 400)
A, B, C = tk.DoubleVar(value=1.0), tk.DoubleVar(value=1.0), tk.DoubleVar(value=1.0)

# --- Create three matplotlib figures ---
figures, axes, canvases = [], [], []

def create_plot(col, title):
    fig, ax = plt.subplots(figsize=(3.3, 2.3))
    ax.set_title(title)
    ax.grid(True)
    canvas = FigureCanvasTkAgg(fig, master=main_frame)
    canvas.draw()
    canvas.get_tk_widget().grid(row=0, column=col, sticky="nsew", padx=5, pady=5)
    figures.append(fig)
    axes.append(ax)
    canvases.append(canvas)

create_plot(0, "y = A·x")
create_plot(1, "y = A·x + B·x²")
create_plot(2, "y = A·x + B·x² + C·x³")

# --- Function to update all plots ---
def update_plots(*args):
    a, b, c = A.get(), B.get(), C.get()
    funcs = [
        a * x,
        a * x + b * x**2,
        a * x + b * x**2 + c * x**3
    ]
    for ax, y in zip(axes, funcs):
        ax.clear()
        ax.plot(x, y, color="blue")
        ax.grid(True)
    axes[0].set_title(f"y = {a:.2f}·x")
    axes[1].set_title(f"y = {a:.2f}·x + {b:.2f}·x²")
    axes[2].set_title(f"y = {a:.2f}·x + {b:.2f}·x² + {c:.2f}·x³")
    for canvas in canvases:
        canvas.draw_idle()

# --- Initial draw ---
update_plots()

# --- Row 1, Column 0: static label ---
label_info = tk.Label(
    main_frame,
    text="Info / Notes\n(Row 1, Col 1)",
    bg="#E0F7FA",
    font=("Arial", 14, "bold"),
    relief="solid"
)
label_info.grid(row=1, column=0, sticky="nsew", padx=5, pady=5)

# --- Row 1, Col 1–2 merged: sliders ---
control_frame = tk.Frame(main_frame, bg="#F3E5F5", relief="solid", bd=1)
control_frame.grid(row=1, column=1, columnspan=2, sticky="nsew", padx=5, pady=5)

control_frame.columnconfigure(0, weight=1)
control_frame.rowconfigure(1, weight=1)

desc = tk.Label(
    control_frame,
    text="Adjust Coefficients A, B, C:",
    bg="#D1C4E9",
    font=("Arial", 12, "bold")
)
desc.pack(fill="x", pady=(10, 5))

slider_frame = tk.Frame(control_frame, bg="#F3E5F5")
slider_frame.pack(expand=True)

# --- Sliders linked to A, B, C ---
sliders = [
    ("A", A),
    ("B", B),
    ("C", C)
]

for name, var in sliders:
    scale = tk.Scale(
        slider_frame,
        variable=var,
        from_=-5,
        to=5,
        resolution=0.1,
        orient="horizontal",
        label=f"Coefficient {name}",
        length=300,
        command=update_plots  # updates on drag
    )
    scale.pack(pady=10)

root.mainloop()
plt.close('all')
